# EDA: social-insurance cases

Appeals in social-insurance cases (`U` → `Ua` / `AUa`): disputes with ZUS about pensions, benefits and contributions, decided in labour and social-insurance divisions.

Every row is one appeal: the first-instance judgment (input) and what the appellate court did with it (`label`: 1 = anything changed, 0 = appeal dismissed). Built by `saos-dataset`; the file hash is in `data/processed/datasets.json`.

In [ ]:
import pandas as pd

from saos_appeals import eda

DOMAIN = "social_insurance"
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

df = eda.load_domain(DOMAIN)
print(eda.dataset_path(DOMAIN).name)
eda.overview(df)

## Target

What the appellate court did. `changed_partly` means the judgment was changed and the appeal dismissed for the rest; `set_aside` usually means a remand.

In [ ]:
eda.outcome_table(df)

In [ ]:
df["remand"].mean().round(3)  # share of rows sent back for re-trial

## Time

Rows per appeal year and the granted share per year. The temporal split (`split`) cuts along this axis, so a drift here lands directly in the test set.

In [ ]:
eda.plot_year(df, f"{DOMAIN}: appeals per year and granted share")

In [ ]:
eda.split_balance(df)

In [ ]:
eda.by_quantile(df, "days_to_appeal")  # time between first-instance judgment and appeal ruling

## Courts

Which courts the rows come from and whether the granted share differs between them. Court identity is a strong, cheap feature, and also a leakage trap: a model that memorises courts will not transfer.

In [ ]:
eda.by_group(df, "first_court_level", min_n=1)

In [ ]:
courts = eda.by_group(df, "first_court_name", min_n=15, top=25)
eda.plot_granted(
    courts,
    f"{DOMAIN}: granted share by first-instance court (n ≥ 15)",
    baseline=df["label"].mean(),
)
courts

In [ ]:
eda.by_group(df, "appeal_court_id", min_n=15).head(15)

## Bench

Number of judges in the first instance (1 = single judge, 3 = with lay judges) and the most frequent presiding judges.

In [ ]:
eda.by_group(df, "judges_n", min_n=1)

In [ ]:
eda.by_group(df, "first_presiding_judge", min_n=8, top=20)

## Texts

Length of the operative part and the reasons of the first-instance judgment, by label. Length is a proxy for case complexity.

In [ ]:
eda.describe_columns(df, ["first_operative_len", "first_reasons_len", "anon_per_1k"])

In [ ]:
eda.plot_hist(df, "first_reasons_len", f"{DOMAIN}: reasons length by label")

In [ ]:
eda.by_quantile(df, "first_reasons_len")

In [ ]:
eda.by_quantile(df, "anon_per_1k")  # anonymisation density: '(...)' per 1,000 chars

## Legal bases and keywords

Acts cited in the first-instance judgment (journal keys of Dziennik Ustaw, named where known) and SAOS keywords. Both are candidate features for a tabular baseline.

In [ ]:
eda.top_items(df, "first_regulation_journals", top=15, names=eda.ACTS)

In [ ]:
eda.top_items(df, "first_regulation_articles", top=20)

In [ ]:
eda.top_items(df, "first_keywords", top=20)

In [ ]:
eda.top_items(df, "first_legal_bases", top=10)

## A row, end to end

One example to keep the schema concrete.

In [ ]:
row = df.iloc[0]
print(row["pair_id"], "|", row["outcome"], "| label", row["label"])
print(row["first_operative_text"][:800])
print("...")
print(row["first_reasons_text"][:800])